In [8]:
%pip install numpy matplotlib

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
import numpy as np 
import matplotlib.pyplot as plt 
import csv

In [10]:

# Função para carregar dados do CSV
def carregar_dados(arquivo):
    X, y = [], []
    with open(arquivo, 'r') as f:
        leitor = csv.reader(f)
        next(leitor) #pula cabeçalho
        for linha in leitor:
            X.append([float(linha[0]), float(linha[1])]) # idade, salario
            y. append(int(linha[2]))
        return np.array(X), np.array(y)

# Carregando o dataset
X, y = carregar_dados("dataset.csv")
print("Formato dos dados:", X.shape, y.shape)
print("Primeiras amostras:\n", X[:5], "\nRótulos:", y[:5])

Formato dos dados: (10, 2) (10,)
Primeiras amostras:
 [[  22. 2000.]
 [  25. 2200.]
 [  47. 4300.]
 [  52. 4600.]
 [  46. 4100.]] 
Rótulos: [0 0 1 1 1]


In [11]:
def entropeia(y):
    classes, contagem = np.unique(y, return_counts=True)
    probs = contagem / len(y)
    return -np.sum(probs * np.log2(probs + 1e-9))

# Teste da entropeia
print("Entropeia [0,0,0,0] ->", entropeia([0,0,0,0]))
print("Entropeia [0,1,0,1] ->", entropeia([0,1,0,1]))

Entropeia [0,0,0,0] -> -1.4426951595367387e-09
Entropeia [0,1,0,1] -> 0.99999999711461


### O QUE A FUNÇÃO DE ENTROPIA FAZ?

##### Na árvore de decisão, precisamos de um critério para medir o quão "puro" ou "misturado" está um conjunto de dados em relação às classes.
• Dois critérios muito usados são:
– Índice Gini (que você já viu e até implementou).
– Entropia (baseada na teoria da informação de Shannon).
• Interpretação:
– Entropia = 0 onde o conjunto totalmente puro (todos os exemplos da mesma classe).
– Entropia alta = onde o conjunto mais misturado (ex.: 50% de uma classe e 50% de outra →
entropia máxima de 1).

• A função entropia mede a desordem/impureza de um conjunto de dados e guia a
árvore de decisão para escolher divisões que deixem os nós mais “puros” possíveis.

In [12]:
def dividir(X, y, feature, limiar):
    esquerda_idx = X[:, feature] <= limiar
    direita_idx = X[:, feature] > limiar
    return X[esquerda_idx], y[esquerda_idx], X[direita_idx], y[direita_idx]

# Testando divisao por idade <= 40
Xe, ye, Xd, yd = dividir(X, y, feature=0, limiar=40)
print("Esquerda:", ye)
print("Direita:", yd)

Esquerda: [0 0 0 0 1]
Direita: [1 1 1 1 1]


In [14]:
class ArvoreDecisao:
    def __init__(self, max_profundidade=3, min_amostras=2):
        self.max_profundidade = max_profundidade
        self.min_amostras = min_amostras
        self.raiz = None

    def ajustar(self, X, y):
        self.raiz = self._crescer(X, y, profundidade=0)

    def _crescer(self, X, y, profundidade):
        num_amostras, num_features = X.shape
        num_classes = len(np.unique(y))
        
        # Condições de parada
        if (profundidade >= self.max_profundidade or
            num_classes == 1 or
            num_amostras < self.min_amostras):
            classe_final = np.bincount(y).argmax()
            return No(valor=classe_final)

        melhor_feature, melhor_limiar, melhor_info = None, None, -1
        melhor_divisao = None

        for feature in range(num_features):
            valores = np.unique(X[:, feature])
            for limiar in valores:
                X_esq, y_esq, X_dir, y_dir = dividir(X, y, feature, limiar)

                if len(y_esq) == 0 or len(y_dir) == 0:
                    continue

                ganho = entropia(y) - (
                    len(y_esq)/num_amostras * entropia(y_esq) +
                    len(y_dir)/num_amostras * entropia(y_dir)
                )

                if ganho > melhor_info:
                    melhor_feature, melhor_limiar, melhor_info = feature, limiar, ganho
                    melhor_divisao = (X_esq, y_esq, X_dir, y_dir)

        if melhor_info == -1:
            classe_final = np.bincount(y).argmax()
            return No(valor=classe_final)

        esquerda = self._crescer(melhor_divisao[0], melhor_divisao[1], profundidade+1)
        direita = self._crescer(melhor_divisao[2], melhor_divisao[3], profundidade+1)

        return No(melhor_feature, melhor_limiar, esquerda, direita)

    def prever(self, X):
        return np.array([self._prever_amostra(x, self.raiz) for x in X])

    def _prever_amostra(self, x, no):
        if no.valor is not None:
            return no.valor
        if x[no.feature] <= no.limiar:
            return self._prever_amostra(x, no.esquerda)
        else:
            return self._prever_amostra(x, no.direita)



